# CrowdFlow Traffic ML Model Training

This Colab notebook trains the machine-learning layer for **CrowdFlow**, an AI-powered adaptive traffic network optimizer.

It covers dataset loading, data cleaning, preprocessing, feature engineering, model training, cross-validation, prediction code, metrics, and presentation-ready graphs.

## What This Model Predicts

- **Regression task:** predict next-hour traffic volume.
- **Classification task:** predict whether the next hour will be congested.
- **Main inputs:** time, weather, holiday, rush-hour flags, lagged traffic, and rolling traffic trends.
- **Algorithms used:** Linear Regression, SGD Gradient Descent, Logistic Regression, Random Forest, and Gradient Boosting.

In [ ]:
# Colab setup: clone the GitHub repo if this notebook is opened directly from Colab.
import os
from pathlib import Path

REPO_URL = "https://github.com/harshiththummala08-sys/Crowd-Flow.git"
REPO_DIR = Path("/content/Crowd-Flow") if Path("/content").exists() else Path.cwd()

if not Path("ml/traffic_ml_pipeline.py").exists():
    if not REPO_DIR.exists():
        !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)
else:
    os.chdir(Path.cwd())

print("Working directory:", Path.cwd())

In [ ]:
# Install/upgrade the small ML stack used by the project.
!python -m pip -q install pandas numpy scikit-learn matplotlib seaborn joblib

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay

from ml.traffic_ml_pipeline import (
    clean_traffic_data,
    engineer_features,
    fit_models,
    get_feature_columns,
    load_dataset,
    predict_next_hour,
    save_artifacts,
    train_test_split_time,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (11, 5)

## 1. Load Dataset

The notebook uses the UCI Metro Interstate Traffic Volume dataset. If the dataset server is unavailable during a live demo, the project creates a deterministic fallback dataset so the workflow still runs.

In [ ]:
raw_df = load_dataset("data/raw")
print(raw_df.shape)
raw_df.head()

## 2. Data Cleaning

Cleaning steps: parse timestamps, remove invalid rows, remove duplicates, normalize numeric columns, convert temperature to Celsius, and clip impossible weather/traffic values.

In [ ]:
clean_df = clean_traffic_data(raw_df)
print(clean_df.shape)
clean_df[["date_time", "temp_c", "rain_1h", "snow_1h", "clouds_all", "weather_main", "traffic_volume"]].head()

In [ ]:
missing_summary = clean_df.isna().mean().sort_values(ascending=False).head(12).to_frame("missing_rate")
missing_summary

## 3. Feature Engineering

The model needs more than raw weather and time. We add rush-hour flags, weekend flags, cyclic time encodings, weather severity, traffic lag features, and rolling traffic trend features.

In [ ]:
model_df = engineer_features(clean_df)
feature_columns, numeric_features, categorical_features = get_feature_columns(model_df)

print("Rows after lag/rolling feature engineering:", len(model_df))
print("Feature count:", len(feature_columns))
print("Congestion threshold:", round(model_df["target_next_hour_volume"].quantile(0.75), 2))
model_df[["date_time", "traffic_volume", "traffic_lag_1h", "traffic_lag_24h", "traffic_roll_mean_6h", "target_next_hour_volume", "target_congested_next_hour"]].head()

## 4. Exploratory Graphs For PPT

These show how traffic changes with time, rush hours, weather, and input features.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(model_df["traffic_volume"], bins=40, kde=True, ax=axes[0], color="#2f80ed")
axes[0].set_title("Traffic Volume Distribution")
sns.boxplot(data=model_df, x="is_rush_hour", y="traffic_volume", ax=axes[1], palette=["#8ecae6", "#ffb703"])
axes[1].set_title("Traffic Volume: Normal vs Rush Hour")
axes[1].set_xticklabels(["Normal", "Rush Hour"])
plt.tight_layout()
plt.show()

In [ ]:
hourly = model_df.groupby("hour")["traffic_volume"].mean().reset_index()
weather = model_df.groupby("weather_main")["traffic_volume"].mean().sort_values(ascending=False).head(10).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.lineplot(data=hourly, x="hour", y="traffic_volume", marker="o", ax=axes[0], color="#1b9aaa")
axes[0].set_title("Average Traffic By Hour")
axes[0].set_xticks(range(0, 24, 2))
sns.barplot(data=weather, y="weather_main", x="traffic_volume", ax=axes[1], palette="viridis")
axes[1].set_title("Average Traffic By Weather")
plt.tight_layout()
plt.show()

In [ ]:
corr_cols = [
    "traffic_volume", "target_next_hour_volume", "temp_c", "rain_1h", "snow_1h", "clouds_all",
    "hour", "is_weekend", "is_rush_hour", "traffic_lag_1h", "traffic_lag_24h", "traffic_roll_mean_6h"
]
corr = model_df[corr_cols].corr(numeric_only=True)
plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Feature Relationship Heatmap")
plt.tight_layout()
plt.show()

## 5. Train Models

Regression models predict traffic volume. Classification models predict congestion. The pipeline includes both simple linear models and stronger non-linear models.

In [ ]:
bundle = fit_models(model_df)
artifact_dir = save_artifacts(bundle, "artifacts/ml")
print("Artifacts saved to:", artifact_dir)
print("Best regression model:", bundle.metrics["best_regression_model"])
print("Best classification model:", bundle.metrics["best_classification_model"])

## 6. Model Metrics

Use these tables directly in the PPT. Regression metrics explain prediction error. Classification metrics explain congestion detection quality.

In [ ]:
regression_metrics = pd.DataFrame(bundle.metrics["regression_holdout"]).T.sort_values("rmse")
regression_metrics

In [ ]:
classification_metrics = pd.DataFrame({
    name: {k: v for k, v in scores.items() if k in ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]}
    for name, scores in bundle.metrics["classification_holdout"].items()
}).T.sort_values("f1", ascending=False)
classification_metrics

In [ ]:
pd.DataFrame(bundle.metrics["cross_validation"]).T

## 7. Prediction Graphs

These plots compare actual vs predicted traffic, residual error, and congestion-classification quality.

In [ ]:
train_df, test_df = train_test_split_time(model_df)
X_test = test_df[bundle.feature_columns]
y_reg_test = test_df["target_next_hour_volume"]
y_cls_test = test_df["target_congested_next_hour"]

reg_pred = bundle.regression_model.predict(X_test)
cls_pred = bundle.classification_model.predict(X_test)
cls_score = bundle.classification_model.predict_proba(X_test)[:, 1]

compare = pd.DataFrame({"actual": y_reg_test.values, "predicted": reg_pred}, index=test_df["date_time"])
compare.head()

In [ ]:
sample = compare.tail(180)
plt.figure(figsize=(15, 5))
plt.plot(sample.index, sample["actual"], label="Actual traffic", linewidth=2)
plt.plot(sample.index, sample["predicted"], label="Predicted traffic", linewidth=2)
plt.title("Actual vs Predicted Next-Hour Traffic Volume")
plt.ylabel("Traffic volume")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(x=y_reg_test, y=reg_pred, alpha=0.35, ax=axes[0], color="#2a9d8f")
axes[0].plot([y_reg_test.min(), y_reg_test.max()], [y_reg_test.min(), y_reg_test.max()], "--", color="#333")
axes[0].set_title("Predicted vs Actual")
axes[0].set_xlabel("Actual")
axes[0].set_ylabel("Predicted")
residuals = y_reg_test - reg_pred
sns.histplot(residuals, bins=40, kde=True, ax=axes[1], color="#e76f51")
axes[1].set_title("Residual Error Distribution")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ConfusionMatrixDisplay.from_predictions(y_cls_test, cls_pred, ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Confusion Matrix")
RocCurveDisplay.from_predictions(y_cls_test, cls_score, ax=axes[1], color="#2f80ed")
axes[1].set_title("ROC Curve")
PrecisionRecallDisplay.from_predictions(y_cls_test, cls_score, ax=axes[2], color="#f77f00")
axes[2].set_title("Precision-Recall Curve")
plt.tight_layout()
plt.show()

## 8. Prediction Code Example

This is the simple prediction call that can be used from backend code after training artifacts are saved.

In [ ]:
example_row = model_df[bundle.feature_columns].tail(1)
prediction = predict_next_hour(bundle, example_row)
prediction

## PPT Explanation Points

- **Fixed signals** use the same timing even when road demand changes.
- **Adaptive control** uses live queue, speed, wait time, and predicted congestion to adjust green time.
- **Gradient descent models** learn coefficients by repeatedly reducing prediction error.
- **Linear regression** gives an explainable traffic-volume baseline.
- **Logistic regression/SGD classifier** converts traffic conditions into congestion probability.
- **Random forest and gradient boosting** capture non-linear rush-hour, weather, and lag interactions.
- **Precision** answers: when the model predicts congestion, how often is it right?
- **Recall** answers: how much real congestion does the model catch?
- **F1 score** balances precision and recall.
- **ROC-AUC** measures how well the model separates congested and non-congested hours across thresholds.
- **Cross-validation** checks model stability across multiple splits instead of trusting one split.